# Setup & Training

In [ ]:
!pip install -U transformers 
!pip install -U datasets 
!pip install -U accelerate 
!pip install -U peft 
!pip install -U trl==0.25.0
!pip install -U bitsandbytes 
!pip install -U wandb
!pip install protobuf==3.20.3
print("Finished")

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training,
    get_peft_model,
)
import os, torch, wandb
from datasets import load_dataset
from trl import SFTTrainer, setup_chat_format

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("Hugging Face")

login(token = hf_token)

wb_token = user_secrets.get_secret("Weights & Biases")

wandb.login(key=wb_token)
run = wandb.init(
    project='Fine-tune Llama 3 8B for TU Dresden Project', 
    job_type="training", 
    anonymous="allow"
)

In [ ]:
import torch
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "dramiley/llama3-8b-finetuned-new"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    use_cache=False
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, peft_config)

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
from datasets import Dataset
from datasets import Dataset

file_path = "train_dataset_mcq.csv" 

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "robinmorgenstern/tudresdenprojekt",
  file_path,
)

df2 = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "robinmorgenstern/tudresdenprojekt",
  "train_dataset_saq.csv",
)

dataset = Dataset.from_pandas(df)
dataset2 = Dataset.from_pandas(df2)

num_samples = min(1000, len(dataset))
dataset = dataset.shuffle(seed=65).select(range(num_samples))
num_samples2 = min(1000, len(dataset2))
dataset2 = dataset2.shuffle(seed=65).select(range(num_samples2))

dataset_dict = dataset.train_test_split(test_size=0.1)
dataset_dict2 = dataset2.train_test_split(test_size=0.1)

train_dataset = dataset_dict['train']
test_dataset = dataset_dict['test']

train_dataset2 = dataset_dict2['train']
test_dataset2 = dataset_dict2['test']

print(train_dataset)
print(train_dataset2)

In [ ]:
def formatting_prompts_func(example):
    output_texts = []
    
    prompts = example['prompt'] if isinstance(example['prompt'], list) else [example['prompt']]
    answers = example['answer_idx'] if isinstance(example['answer_idx'], list) else [example['answer_idx']]
    
    for p, a in zip(prompts, answers):
        clean_p = str(p).replace('Provide as JSON format: {"answer_choice":""}', "").strip()
        
        text = f"Question:\n{clean_p}\n{a}{tokenizer.eos_token}"
        output_texts.append(text)
        
    return output_texts

In [ ]:
import ast

def formatting_prompts_func2(example):
    output_texts = []
    
    prompts = example['en_question'] if isinstance(example['en_question'], list) else [example['en_question']]
    annotations_raw = example['annotations'] if isinstance(example['annotations'], list) else [example['annotations']]
    
    for p, raw_ann in zip(prompts, annotations_raw):
        
        annotations = raw_ann
        if isinstance(raw_ann, str):
            try:
                annotations = ast.literal_eval(raw_ann)
            except:
                annotations = []
        
        clean_p = str(p).replace('Provide as JSON format: {"answer_choice":""}', "").strip()
        clean_p = clean_p.replace('Without any explanation, choose only one from the given alphabet choices(e.g., A, B, C).', "").strip()
        
        best_answer = "unknown"
        
        if isinstance(annotations, list) and len(annotations) > 0:
            for entry in annotations:
                if not isinstance(entry, dict): continue
                
                if "en_answers" in entry and entry["en_answers"]:
                    ans_list = entry["en_answers"]
                    if isinstance(ans_list, list) and len(ans_list) > 0:
                        best_answer = str(ans_list[0])
                        break
                
                elif "answers" in entry and entry["answers"]:
                    ans_list = entry["answers"]
                    if isinstance(ans_list, list) and len(ans_list) > 0:
                        best_answer = str(ans_list[0])
                        break

        text = f"Question:\n{clean_p}\n\nAnswer:\n{best_answer}{tokenizer.eos_token}"
        output_texts.append(text)
        
    return output_texts


In [ ]:
train_dataset = train_dataset.map(
    lambda x: {"text": formatting_prompts_func(x)},
    batched=True
)

test_dataset = test_dataset.map(
    lambda x: {"text": formatting_prompts_func(x)},
    batched=True
)


train_dataset2 = train_dataset2.map(
    lambda x: {"text": formatting_prompts_func2(x)},
    batched=True
)

test_dataset2 = test_dataset2.map(
    lambda x: {"text": formatting_prompts_func2(x)},
    batched=True
)
print(train_dataset["text"][0])
print(train_dataset2["text"][0])

In [ ]:
import re
from datasets import concatenate_datasets

def convert_mcq_to_saq_stripped(example):
    prompt = example['prompt']
    clean_prompt_base = str(prompt).replace('Without any explanation, choose only one from the given alphabet choices(e.g., A, B, C). Provide as JSON format: {"answer_choice":""}', "").strip()
    
    if 'answer_idx' not in example or not example['answer_idx']:
        return {"text": None}

    correct_letter = example['answer_idx'].strip().upper()
    
    pattern = rf"{correct_letter}[\.\)]\s*(.*?)(?:\s[A-D][\.\)]|$)"
    match = re.search(pattern, clean_prompt_base, re.DOTALL)
    
    if not match:
        return {"text": None}
    
    answer_text = match.group(1).strip()
    
    if not answer_text or answer_text == ".":
        return {"text": None}
    
    if "A." in clean_prompt_base:
        question_only = clean_prompt_base.split("A.")[0].strip()
    elif "(A)" in clean_prompt_base:
        question_only = clean_prompt_base.split("(A)")[0].strip()
    elif "A)" in clean_prompt_base:
        question_only = clean_prompt_base.split("A)")[0].strip()
    else:
        question_only = clean_prompt_base

    question_only = question_only.strip()

    new_text = f"Question:\n{question_only}\n\nAnswer:\n{answer_text}{tokenizer.eos_token}"
    
    return {"text": new_text}

print("Erstelle künstliche SAQ Daten...")

saq_augmented_data = train_dataset.map(convert_mcq_to_saq_stripped)

saq_augmented_data = saq_augmented_data.filter(lambda x: x["text"] is not None)

print(f"Augmented Daten erstellt: {len(saq_augmented_data)} Beispiele.")

print(saq_augmented_data["text"][0])
print(saq_augmented_data["text"][1])

dataset_dict = saq_augmented_data.train_test_split(test_size=0.1, seed=42)
saq_aug_train = dataset_dict['train']
saq_aug_test = dataset_dict['test']

print("Füge alles zusammen...")

combined_train = concatenate_datasets([
    train_dataset.select_columns(["text"]),  # Original MCQ Train
    train_dataset2.select_columns(["text"]), # Original SAQ Train
    saq_aug_train.select_columns(["text"])   # Augmented SAQ Train
])

combined_test = concatenate_datasets([
    test_dataset.select_columns(["text"]),   # Original MCQ Test
    test_dataset2.select_columns(["text"]),  # Original SAQ Test
    saq_aug_test.select_columns(["text"])    # Augmented SAQ Test
])
combined_train = combined_train.filter(lambda x: x["text"] is not None)
combined_test = combined_test.filter(lambda x: x["text"] is not None)

train_dataset = combined_train.shuffle(seed=42)
test_dataset = combined_test.shuffle(seed=42)

print(f"Fertiges Training-Set: {len(train_dataset)} Beispiele.")
print(f"Fertiges Test-Set: {len(test_dataset)} Beispiele.")
print(f"Spalten: {train_dataset.column_names}")

print("-" * 20)
print("Beispiel Text:")
print(train_dataset[0]['text'][:200])

In [ ]:
def flatten_text_column(example):
    text = example["text"]
    if isinstance(text, list):
        return {"text": text[0]}
    
    return {"text": text}

print("Repariere Dataset-Struktur...")
train_dataset = train_dataset.map(flatten_text_column)
test_dataset = test_dataset.map(flatten_text_column)

print(f"Check Typ: {type(train_dataset[0]['text'])}")
print(f"Check Inhalt: {train_dataset[0]['text'][:50]}...")

print(train_dataset["text"][0])

In [ ]:
sft_config = SFTConfig(
    output_dir="llama3-8b-finetuned-new",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=2,
    fp16=True,
    bf16=False,
    report_to="wandb",
    completion_only_loss = False,
    warmup_ratio=0.1,
    max_grad_norm=0.5,
    eval_strategy="steps",
    eval_steps=10,
)

sft_config.max_seq_length = 1024
sft_config.packing = False
sft_config.dataset_text_field = "text"


trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=sft_config,
)


print("Starte Training nächste Runde ...")
trainer.train()


new_adapter_path = "llama3-8b-finetune-adapters"
trainer.model.save_pretrained(new_adapter_path)
tokenizer.save_pretrained(new_adapter_path)


In [ ]:
wandb.finish()
model.config.use_cache = True

In [ ]:
new_model = "llama3-8b-finetuned"
trainer.model.save_pretrained(new_model)
trainer.model.push_to_hub(new_model, use_temp_dir=False)
tokenizer.save_pretrained(new_model)
tokenizer.push_to_hub(new_model, use_temp_dir=False)

In [ ]:
import torch
import gc

# to free up GPU space –> deleting the model and trainer objects
try:
    del model
    del trainer
except NameError:
    pass


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

new_model = "llama3-8b-finetuned"

base_model_id = "dramiley/llama3-8b-finetuned-new" 

print(f"Lade Basis-Modell: {base_model_id}")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="cpu",
    offload_folder="offload",
    low_cpu_mem_usage=True,
)

tokenizer = AutoTokenizer.from_pretrained(new_model)

print(f"Lade Adapter: {new_model}")
model = PeftModel.from_pretrained(base_model, new_model)

print("Merging model...")
model = model.merge_and_unload()

if hasattr(model, "peft_config"):
    print("Entferne verwaiste PEFT-Konfiguration...")
    del model.peft_config
    
if hasattr(model, "_hf_peft_config_loaded"):
    del model._hf_peft_config_loaded


print("Speichere Full Model...")
model.save_pretrained(new_model)
tokenizer.save_pretrained(new_model)


print("Upload...")
model.push_to_hub(new_model, use_temp_dir=False)
tokenizer.push_to_hub(new_model, use_temp_dir=False)

print("Fertig!")

# Evaluation

In [ ]:
!pip install -U sentence-transformers
!pip install -U transformers 
!pip install -U datasets 
!pip install -U accelerate 
!pip install -U peft 
!pip install -U trl==0.25.0
!pip install -U bitsandbytes 
!pip install -U wandb
!pip install protobuf==3.20.3
print("Finished")

# MCQ

In [ ]:
import torch
import gc
import os
import re
import pandas as pd
import zipfile
from tqdm import tqdm
from IPython.display import FileLink

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, util
import kagglehub
from kagglehub import KaggleDatasetAdapter

print("Bereinige Speicher...")
gc.collect()
torch.cuda.empty_cache()


model_id = "dramiley/llama3-8b-finetuned-new"
print(f"Lade LLM: {model_id}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
model.eval()

print("Lade Sentence-Transformer...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Baue Knowledge Base aus MCQ-Trainingsdaten...")
try:
    df_train = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "robinmorgenstern/tudresdenprojekt", "train_dataset_mcq.csv")
except:
    train_path = "train_dataset_mcq.csv"
    for root, dirs, files in os.walk("."):
        for file in files:
            if "train" in file and "mcq" in file and file.endswith(".csv"):
                train_path = os.path.join(root, file)
    df_train = pd.read_csv(train_path)

knowledge_base = []
junk_regex = re.compile(r'(Provide as JSON format.*?)|(Without any explanation.*?)|((:?\s*\{"answer_choice":\s*""\}))', re.IGNORECASE | re.DOTALL)

for _, row in tqdm(df_train.iterrows(), total=len(df_train)):
    if pd.isna(row.get('answer_idx')): continue
    
    raw_prompt = str(row['prompt'])
    
    clean_text = junk_regex.sub("", raw_prompt).strip()
    
    question_only = clean_text
    if "A." in question_only: 
        question_only = question_only.split("A.")[0].strip()
    elif "(A)" in question_only:
        question_only = question_only.split("(A)")[0].strip()
        
    example_body = re.sub(r'\s*Answer:\s*$', '', clean_text, flags=re.IGNORECASE).strip()
    correct_char = row['answer_idx'].strip().upper()
    
    full_example = f"Question:\n{example_body}\n\nAnswer:\n{correct_char}\n"
    
    if len(question_only) > 5:
        knowledge_base.append({
            "embedding_text": question_only,
            "full_text": full_example
        })

print(f"Knowledge Base Größe: {len(knowledge_base)} Einträge.")

kb_texts = [entry["embedding_text"] for entry in knowledge_base]
kb_embeddings = embedder.encode(kb_texts, convert_to_tensor=True).to("cpu")

print("Lade MCQ Test-Daten...")
try:
    df_test = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "robinmorgenstern/tudresdenprojekt", "test_dataset_mcq.csv")
except:
    test_path = "test_dataset_mcq.csv"
    for root, dirs, files in os.walk("."):
        for file in files:
            if "test" in file and "mcq" in file and file.endswith(".csv"):
                test_path = os.path.join(root, file)
    df_test = pd.read_csv(test_path)

print("Starte Dynamic Few-Shot Inference...")

predictions = []
ids = []

options = ["A", "B", "C", "D"]
ids_space = [tokenizer.encode(" " + x, add_special_tokens=False)[-1] for x in options]
ids_raw = [tokenizer.encode(x, add_special_tokens=False)[-1] for x in options]

with torch.no_grad():
    for index, row in tqdm(df_test.iterrows(), total=len(df_test)):
        
        curr_id = row['MCQID'] if 'MCQID' in row else row['id']
        ids.append(curr_id)
        
        raw_prompt = str(row['prompt'])
        clean_text = junk_regex.sub("", raw_prompt).strip()
        
        q_search = clean_text
        if "A." in q_search: q_search = q_search.split("A.")[0].strip()
        
        clean_prompt = re.sub(r'\s*Answer:\s*$', '', clean_text, flags=re.IGNORECASE).strip()
        
        query_emb = embedder.encode(q_search, convert_to_tensor=True).to("cpu")
        hits = util.semantic_search(query_emb, kb_embeddings, top_k=3)[0]
        
        prompt_text = "Answer the multiple choice question by selecting the correct letter (A, B, C, or D).\n\n"
        
        for hit in hits:
            idx = hit['corpus_id']
            prompt_text += f"{knowledge_base[idx]['full_text']}\n\n"
            
        prompt_text += f"Question:\n{clean_prompt}\n\nAnswer:\n"

        inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
        outputs = model(**inputs)
        
        logits = outputs.logits[0, -1, :]
        
        best_char = "C" # Fallback
        best_score = -float('inf')
        
        for i, char in enumerate(options):
            score = logits[ids_space[i]].item() + logits[ids_raw[i]].item()
            
            if score > best_score:
                best_score = score
                best_char = char
                
        predictions.append(best_char)


print("Speichere Ergebnisse...")

mcq_rows = []
for mcq_id, pred in zip(ids, predictions):
    row = {
        "MCQID": mcq_id, 
        "A": "True" if pred == "A" else "False",
        "B": "True" if pred == "B" else "False",
        "C": "True" if pred == "C" else "False",
        "D": "True" if pred == "D" else "False"
    }
    mcq_rows.append(row)

df_sub = pd.DataFrame(mcq_rows)
df_sub.to_csv("mcq_prediction.tsv", sep='\t', index=False)


zip_name = "submission_mcq_rag.zip"
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write("mcq_prediction.tsv", arcname="mcq_prediction.tsv")
    
    if os.path.exists("saq_prediction.tsv"):
        print("Füge vorhandene SAQ-Datei hinzu...")
        zipf.write("saq_prediction.tsv", arcname="saq_prediction.tsv")

print(f"{zip_name} fertig!")
FileLink(zip_name)

# SAQ

In [ ]:
import ast
import re
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sentence_transformers import SentenceTransformer


print("Baue Knowledge Base aus Trainingsdaten...")

knowledge_base = []

try:
    df_train_mcq = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "robinmorgenstern/tudresdenprojekt", "train_dataset_mcq.csv")
    df_train_saq = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "robinmorgenstern/tudresdenprojekt", "train_dataset_saq.csv")
except:
    df_train_mcq = pd.read_csv("train_dataset_mcq.csv") 
    df_train_saq = pd.read_csv("train_dataset_saq.csv")

def extract_answer_text(prompt, letter):
    if not isinstance(letter, str): return None
    pattern = rf"{letter}[\.\)]\s*(.*?)(?:\s[A-D][\.\)]|$)"
    match = re.search(pattern, prompt, re.DOTALL)
    return match.group(1).strip() if match else None

junk_regex = re.compile(
    r'([.]?\s*Provide as JSON format.*?)'
    r'|(Without any explanation.*?)'
    r'|((:?\s*\{"answer_choice":\s*""\}))'
    r'|(:?\s*choose only one from the given alphabet choices\(e\.g\., A, B, C\):?\s*)', 
    re.IGNORECASE | re.DOTALL
)

print("Lade Sentence-Transformer...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Verarbeite {len(df_train_mcq)} MCQ-Zeilen (mit Deep Cleaning)...")

for _, row in df_train_mcq.iterrows():
    if pd.isna(row.get('answer_idx')): continue
    
    raw_prompt = str(row.get('prompt', ''))
    clean_q = junk_regex.sub("", raw_prompt)
    
    if "A." in clean_q: clean_q = clean_q.split("A.")[0].strip()
    elif "(A)" in clean_q: clean_q = clean_q.split("(A)")[0].strip()
    
    clean_q = re.sub(r'\s*Answer:\s*$', '', clean_q, flags=re.IGNORECASE).strip()

    clean_q = clean_q.strip(" ,")
    
    ans_text = extract_answer_text(raw_prompt, row['answer_idx'])
    
    if ans_text:
        ans_text = junk_regex.sub("", ans_text)
        ans_text = ans_text.replace('{"answer_choice":""}', "")
        ans_text = ans_text.replace('Answer:', "")
        ans_text = ans_text.strip(" .,\n\r")
    
    if ans_text and len(ans_text) > 0 and len(clean_q) > 5:
        knowledge_base.append({"q": clean_q, "a": ans_text})
        
        if "JSON" in ans_text:
            print(f"WARNUNG: Immer noch Müll in ID {row.get('id', '?')}: {ans_text}")

print(f"Verarbeite {len(df_train_saq)} SAQ-Zeilen...")

q_col = 'en_question' if 'en_question' in df_train_saq.columns else 'prompt'
a_col = 'annotations' if 'annotations' in df_train_saq.columns else 'answers'

for _, row in df_train_saq.iterrows():
    raw_q = str(row.get(q_col, ""))
    clean_q = junk_regex.sub("", raw_q).replace("Answer:", "").strip()
    
    raw_a = row.get(a_col, [])
    best_answer = None
    
    try:
        if isinstance(raw_a, str):
            raw_a = ast.literal_eval(raw_a)
            
        if isinstance(raw_a, list) and len(raw_a) > 0:
            entry = raw_a[0]
            if isinstance(entry, dict):
                if 'en_answers' in entry and entry['en_answers']:
                    best_answer = entry['en_answers'][0]
                elif 'answers' in entry and entry['answers']:
                    best_answer = entry['answers'][0]
    except:
        continue

    if best_answer and len(clean_q) > 5:
        knowledge_base.append({"q": clean_q, "a": str(best_answer)})

print(f"Knowledge Base Größe: {len(knowledge_base)} Beispiele.")

kb_questions = [entry["q"] for entry in knowledge_base]
kb_embeddings = embedder.encode(kb_questions, convert_to_tensor=True).to("cpu")

In [ ]:
import torch
import pandas as pd
import zipfile
import os
import re
import string
import kagglehub
from kagglehub import KaggleDatasetAdapter
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import util


def normalize_answer(s):
    s = str(s).lower()
    exclude = set(string.punctuation)
    s = ''.join(ch for ch in s if ch not in exclude)
    prefixes = ["the ", "a ", "an "]
    for p in prefixes:
        if s.startswith(p):
            s = s[len(p):]
    return s.strip()


print("Lade SAQ Test-Daten...")

try:
    df_test_saq = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "robinmorgenstern/tudresdenprojekt", "test_dataset_saq.csv")
except:
    test_path = "test_dataset_saq.csv"
    for root, dirs, files in os.walk("."):
        for file in files:
            if "test" in file and "saq" in file and file.endswith(".csv"):
                test_path = os.path.join(root, file)
    df_test_saq = pd.read_csv(test_path)

id_col = 'ID' if 'ID' in df_test_saq.columns else 'id'
q_col = 'prompt' if 'prompt' in df_test_saq.columns else 'en_question'

print(f"Test-Set geladen: {len(df_test_saq)} Fragen.")

model_id = "dramiley/llama3-8b-finetuned-new"

print(f"Lade Modell: {model_id}...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
model.eval()


predictions = []
ids = []

stop_tokens = [
    tokenizer.eos_token_id, 
    tokenizer.convert_tokens_to_ids("<|eot_id|>"),
    tokenizer.encode("\n", add_special_tokens=False)[-1]
]

print("Starte Dynamic Few-Shot SAQ...")

with torch.no_grad():
    for index, row in tqdm(df_test_saq.iterrows(), total=len(df_test_saq)):
        
        curr_id = row[id_col]
        ids.append(curr_id)
        
        raw_prompt = str(row[q_col])
        test_q = junk_regex.sub("", raw_prompt).strip()
        test_q = re.sub(r'\s*Answer:\s*$', '', test_q, flags=re.IGNORECASE).strip()
        
        query_emb = embedder.encode(test_q, convert_to_tensor=True).to("cpu")
        
        hits = util.semantic_search(query_emb, kb_embeddings, top_k=3)[0]
           
        prompt_text = "Answer the question concisely with a single entity, name, or number.\n\n"
        
        for hit in hits:
            idx = hit['corpus_id']
            example = knowledge_base[idx]
            prompt_text += f"Question:\n{example['q']}\n\nAnswer:\n{example['a']}\n\n"
        
        prompt_text += f"Question:\n{test_q}\n\nAnswer:\n"

        inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
        
        outputs = model.generate(
            **inputs,
            max_new_tokens=15,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=stop_tokens
        )
        
        decoded = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        answer = decoded.strip().split('\n')[0].strip()
        
        answer = normalize_answer(answer)
        if answer.endswith("."): answer = answer[:-1]
        
        if not answer: answer = "unknown"
        
        predictions.append(answer)


print("Erstelle Submission...")

submission_data = [{"ID": i, "answer": a} for i, a in zip(ids, predictions)]
df_sub = pd.DataFrame(submission_data)

df_sub.to_csv("saq_prediction.tsv", sep='\t', index=False)

zip_name = "submission_rag_final.zip"
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write("saq_prediction.tsv", arcname="saq_prediction.tsv")
    
    if os.path.exists("mcq_prediction.tsv"):
        print("Füge vorhandene MCQ Datei hinzu...")
        zipf.write("mcq_prediction.tsv", arcname="mcq_prediction.tsv")

print(f"{zip_name} fertig!")
FileLink(zip_name)